# Snake Game with Reinforcement Learning (Q-Learning)
## Assignment 5

This notebook implements a classic Snake game where an AI agent learns to play using **Q-Learning**, a model-free reinforcement learning algorithm.

### How it works:
- The **agent** observes the game state (danger directions, food direction, current movement)
- It chooses actions (UP, DOWN, LEFT, RIGHT) using an epsilon-greedy policy
- It learns from rewards: +10 for eating food, -10 for dying, 0 otherwise
- Over many episodes, the Q-table grows and the agent improves

## 1. Imports and Constants

In [ ]:
import pygame
import random
import numpy as np
import pickle
import os
from collections import defaultdict
import matplotlib.pyplot as plt

CELL_SIZE = 20
GRID_WIDTH = 30
GRID_HEIGHT = 20
SCREEN_WIDTH = CELL_SIZE * GRID_WIDTH
SCREEN_HEIGHT = CELL_SIZE * GRID_HEIGHT
FPS = 10

BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
GREEN = (0, 200, 0)
DARK_GREEN = (0, 155, 0)
RED = (200, 0, 0)
GRAY = (40, 40, 40)

UP = (0, -1)
DOWN = (0, 1)
LEFT = (-1, 0)
RIGHT = (1, 0)
ACTIONS = [UP, DOWN, LEFT, RIGHT]

Q_FILE = "snake_q_table.pkl"

print("Imports loaded successfully!")
print(f"Grid: {GRID_WIDTH}x{GRID_HEIGHT} | Screen: {SCREEN_WIDTH}x{SCREEN_HEIGHT}")

## 2. Snake Game Environment

In [ ]:
class SnakeGame:
    """Snake game environment for reinforcement learning."""

    def __init__(self):
        self.width = GRID_WIDTH
        self.height = GRID_HEIGHT
        self.reset()

    def reset(self):
        mid_x = self.width // 2
        mid_y = self.height // 2
        self.snake = [(mid_x, mid_y), (mid_x - 1, mid_y), (mid_x - 2, mid_y)]
        self.direction = RIGHT
        self.food = self._place_food()
        self.score = 0
        self.steps = 0
        self.steps_since_food = 0
        return self._get_state()

    def _place_food(self):
        while True:
            pos = (random.randint(0, self.width - 1), random.randint(0, self.height - 1))
            if pos not in self.snake:
                return pos

    def _get_state(self):
        head = self.snake[0]
        dir_x, dir_y = self.direction

        point_straight = (head[0] + dir_x, head[1] + dir_y)
        point_right = (head[0] - dir_y, head[1] + dir_x)
        point_left = (head[0] + dir_y, head[1] - dir_x)

        def is_danger(point):
            return (
                point[0] < 0 or point[0] >= self.width
                or point[1] < 0 or point[1] >= self.height
                or point in self.snake
            )

        food_dx = self.food[0] - head[0]
        food_dy = self.food[1] - head[1]

        state = (
            is_danger(point_straight),    # danger straight
            is_danger(point_right),        # danger right
            is_danger(point_left),         # danger left
            self.direction == UP,          # moving up
            self.direction == DOWN,        # moving down
            self.direction == LEFT,        # moving left
            self.direction == RIGHT,       # moving right
            food_dy < 0,                   # food up
            food_dy > 0,                   # food down
            food_dx < 0,                   # food left
            food_dx > 0,                   # food right
        )
        return state

    def step(self, action_idx):
        self.direction = ACTIONS[action_idx]
        head = self.snake[0]
        dx, dy = self.direction
        new_head = (head[0] + dx, head[1] + dy)
        self.steps += 1
        self.steps_since_food += 1

        # Collision with wall or self = death
        if (
            new_head[0] < 0 or new_head[0] >= self.width
            or new_head[1] < 0 or new_head[1] >= self.height
            or new_head in self.snake
        ):
            return self._get_state(), -10, True

        self.snake.insert(0, new_head)

        if new_head == self.food:
            self.score += 1
            self.steps_since_food = 0
            self.food = self._place_food()
            reward = 10
        else:
            self.snake.pop()
            reward = 0

        # Timeout: snake is stuck
        if self.steps_since_food > 100 * len(self.snake):
            return self._get_state(), -10, True

        return self._get_state(), reward, False

    def render(self, screen):
        screen.fill(BLACK)
        for x in range(0, SCREEN_WIDTH, CELL_SIZE):
            pygame.draw.line(screen, GRAY, (x, 0), (x, SCREEN_HEIGHT))
        for y in range(0, SCREEN_HEIGHT, CELL_SIZE):
            pygame.draw.line(screen, GRAY, (0, y), (SCREEN_WIDTH, y))
        for i, (x, y) in enumerate(self.snake):
            color = DARK_GREEN if i == 0 else GREEN
            rect = pygame.Rect(x * CELL_SIZE, y * CELL_SIZE, CELL_SIZE, CELL_SIZE)
            pygame.draw.rect(screen, color, rect)
            pygame.draw.rect(screen, BLACK, rect, 1)
        fx, fy = self.food
        food_rect = pygame.Rect(fx * CELL_SIZE, fy * CELL_SIZE, CELL_SIZE, CELL_SIZE)
        pygame.draw.rect(screen, RED, food_rect)
        font = pygame.font.SysFont("arial", 20)
        score_text = font.render(f"Score: {self.score}", True, WHITE)
        screen.blit(score_text, (10, 5))


print("SnakeGame class defined.")
print(f"State space: 11 boolean values -> 2^11 = {2**11} possible states")
print(f"Action space: {len(ACTIONS)} actions (UP, DOWN, LEFT, RIGHT)")

## 3. Q-Learning Agent

In [ ]:
class QLearningAgent:
    """Q-Learning agent that learns to play the snake game."""

    def __init__(self, alpha=0.1, gamma=0.95, epsilon=1.0, epsilon_decay=0.995, epsilon_min=0.01):
        self.q_table = defaultdict(lambda: np.zeros(len(ACTIONS)))
        self.alpha = alpha          # Learning rate
        self.gamma = gamma          # Discount factor
        self.epsilon = epsilon      # Exploration rate
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, len(ACTIONS) - 1)  # Explore
        return int(np.argmax(self.q_table[state]))       # Exploit

    def learn(self, state, action, reward, next_state, done):
        best_next = np.max(self.q_table[next_state])
        target = reward + (0 if done else self.gamma * best_next)
        self.q_table[state][action] += self.alpha * (target - self.q_table[state][action])

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def save(self, filepath=Q_FILE):
        with open(filepath, "wb") as f:
            pickle.dump(dict(self.q_table), f)

    def load(self, filepath=Q_FILE):
        if os.path.exists(filepath):
            with open(filepath, "rb") as f:
                data = pickle.load(f)
            self.q_table = defaultdict(lambda: np.zeros(len(ACTIONS)), data)
            return True
        return False


print("QLearningAgent class defined.")
print(f"Hyperparameters: alpha=0.1, gamma=0.95, epsilon_start=1.0, decay=0.995")

## 4. Training Loop

In [ ]:
def train(episodes=1000):
    game = SnakeGame()
    agent = QLearningAgent()

    print("=" * 60)
    print("  TRAINING Q-LEARNING AGENT")
    print("=" * 60)
    print(f"Episodes: {episodes}")
    print(f"Q-table size before: {len(agent.q_table)}")
    print()

    best_score = 0
    scores = []
    avg_scores = []

    for ep in range(1, episodes + 1):
        state = game.reset()
        done = False

        while not done:
            action = agent.choose_action(state)
            next_state, reward, done = game.step(action)
            agent.learn(state, action, reward, next_state, done)
            state = next_state

        agent.decay_epsilon()
        scores.append(game.score)

        if game.score > best_score:
            best_score = game.score

        if ep % 100 == 0:
            avg = sum(scores[-100:]) / len(scores[-100:])
            avg_scores.append(avg)
            print(f"Episode {ep:5d} | Avg Score: {avg:6.1f} | Best: {best_score} | Epsilon: {agent.epsilon:.4f} | Q-states: {len(agent.q_table)}")

    agent.save()
    print()
    print(f"Training complete! Best score: {best_score}")
    print(f"Q-table saved to {Q_FILE} ({len(agent.q_table)} states)")

    return agent, scores, avg_scores


print("train() function defined.")

## 5. Run Training

In [ ]:
EPISODES = 1000
agent, scores, avg_scores = train(EPISODES)

## 6. Training Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot raw scores per episode
axes[0].plot(scores, alpha=0.3, color="blue", label="Score per episode")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Score")
axes[0].set_title("Score per Episode")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot rolling average
axes[1].plot(range(100, EPISODES + 1, 100), avg_scores, color="red", marker="o", label="Avg score (last 100)")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Average Score")
axes[1].set_title("Average Score (rolling 100 episodes)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final Q-table size: {len(agent.q_table)} states")
print(f"Best score achieved: {max(scores)}")
print(f"Average score (last 100): {avg_scores[-1]:.1f}")

## 7. Watch the Trained Agent Play

Run the cell below to open a pygame window and watch the trained agent play the snake game.

**Close the pygame window to stop.**

In [ ]:
def play_with_agent(agent=None):
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("Snake RL - Watch Trained Agent Play")
    clock = pygame.time.Clock()

    game = SnakeGame()
    if agent is None:
        agent = QLearningAgent()
        if not agent.load():
            print("No saved Q-table found. Train first!")
            pygame.quit()
            return

    agent.epsilon = 0  # No exploration, pure exploitation
    state = game.reset()
    running = True
    games_played = 0

    print("Watching trained agent play... (close window to stop)")

    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

        action = agent.choose_action(state)
        state, reward, done = game.step(action)
        game.render(screen)

        font = pygame.font.SysFont("arial", 16)
        info = font.render(
            f"Q-States: {len(agent.q_table)} | Steps: {game.steps} | Games: {games_played}",
            True, WHITE
        )
        screen.blit(info, (10, SCREEN_HEIGHT - 25))
        pygame.display.flip()
        clock.tick(FPS)

        if done:
            games_played += 1
            print(f"Game Over! Score: {game.score} | Steps: {game.steps}")
            pygame.time.wait(1000)
            state = game.reset()

    pygame.quit()
    print(f"Session ended. Games played: {games_played}")


# Uncomment the line below to watch the agent play:
# play_with_agent(agent)

## 8. Manual Play Mode

Play the snake game yourself using arrow keys!

In [ ]:
def play_manual():
    pygame.init()
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("Snake RL - Manual Play")
    clock = pygame.time.Clock()

    game = SnakeGame()
    state = game.reset()
    running = True

    direction_map = {
        pygame.K_UP: UP,
        pygame.K_DOWN: DOWN,
        pygame.K_LEFT: LEFT,
        pygame.K_RIGHT: RIGHT,
    }

    print("Use arrow keys to play. Close window to quit.")

    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key in direction_map:
                    new_dir = direction_map[event.key]
                    opposite = (-game.direction[0], -game.direction[1])
                    if new_dir != opposite:
                        game.direction = new_dir

        state, reward, done = game.step(ACTIONS.index(game.direction))
        game.render(screen)
        pygame.display.flip()
        clock.tick(FPS)

        if done:
            print(f"Game Over! Score: {game.score}")
            pygame.time.wait(1000)
            state = game.reset()

    pygame.quit()


# Uncomment the line below to play manually:
# play_manual()

## Summary

| Component | Description |
|-----------|-------------|
| **Algorithm** | Q-Learning (tabular) |
| **State** | 11 boolean features (danger + direction + food location) |
| **Actions** | 4 (UP, DOWN, LEFT, RIGHT) |
| **Rewards** | +10 (food), -10 (death), 0 (move) |
| **Learning Rate (alpha)** | 0.1 |
| **Discount Factor (gamma)** | 0.95 |
| **Exploration (epsilon)** | 1.0 -> 0.01 (decay: 0.995) |

### Key RL Concepts:
- **Exploration vs Exploitation**: epsilon-greedy policy balances trying new moves vs using known good moves
- **Temporal Difference Learning**: updates Q-values after each step using bootstrapping
- **State Representation**: compact 11-feature vector captures all relevant game info